In [0]:
# ---------------------------------------------------------------- config ---

project_identifier = 'dac003'

# DRY_RUN redirects every write into 8_dev and skips the staging rebuild,
# reading the existing prod staging tables instead. Reads always come from prod.
# It exercises the same view-building code that ships, so a dry run genuinely
# validates the production path.
#
# Default is a real run. Override without editing this notebook by passing the
# job parameter dry_run=true.
try:
    DRY_RUN = dbutils.widgets.get('dry_run').strip().lower() == 'true'
except Exception:
    DRY_RUN = False

REBUILD_STAGING = not DRY_RUN

# Row counts for the manifest are accurate but add several minutes to the run
# (map_pathology_report is a 124M row source). Off by default.
COMPUTE_MANIFEST_COUNTS = False

# Staging is always read from, and in a production run written to, its
# original home. The cohort definition depends on it.
staging_schema = '5_projects.dac003_breastonestop'

if DRY_RUN:
    target_catalog = '8_dev'
    target_schema  = f'{project_identifier}_dryrun'
    cohort_table   = f'8_dev.{project_identifier}_dryrun.cohort'
    bcnb_xwalk     = f'8_dev.{project_identifier}_dryrun.bcnb_person_xwalk'
else:
    target_catalog = '5_projects'
    target_schema  = project_identifier
    cohort_table   = '6_mgmt.cohorts.dac003'
    bcnb_xwalk     = '6_mgmt.cohorts.bcnb_person_xwalk'

target = f'{target_catalog}.{target_schema}'

# Unchanged from the original notebook.
max_ig_risk = 3
max_ig_severity = 2
columns_to_exclude = ['ADC_UPDT', 'full_street_address', 'UPRN', 'LATITUDE',
                      'LONGITUDE', 'match_algorithm', 'match_confidence',
                      'match_quality']

rde_tables = ['rde_aliases', 'rde_all_procedures', 'rde_blobdataset', 'rde_all_diagnosis', 'rde_allergydetails', 'rde_apc_diagnosis', 'rde_apc_opcs', 'rde_ariapharmacy', 'rde_blobdataset', 'rde_cds_apc', 'rde_cds_opa', 'rde_critactivity', 'rde_critopcs', 'rde_critperiod', 'rde_emergencyd', 'rde_encounter', 'rde_family_history', 'rde_iqemo', 'rde_measurements', 'rde_medadmin', 'rde_op_diagnosis', 'rde_opa_opcs', 'rde_pathology', 'rde_patient_demographics', 'rde_pc_diagnosis', 'rde_pc_problems', 'rde_pc_procedures', 'rde_pharmacyorders', 'rde_radiology', 'rde_raw_pathology', 'rde_scr_careplan', 'rde_scr_deftreatment', 'rde_scr_demographics', 'rde_scr_diagnosis', 'rde_scr_investigations', 'rde_scr_pathology', 'rde_scr_referrals', 'rde_scr_trackingcomments',
'rde_mat_nnu_episodes', 'rde_mat_nnu_exam', 'rde_mat_nnu_nccmds', 'rde_measurements', 'rde_medadmin', 'rde_mill_powertrials', 'rde_msds_booking', 'rde_msds_carecontact', 'rde_msds_delivery', 'rde_msds_diagnosis',
'rde_powerforms',]

# The literal above is kept byte-identical to the original for reviewability;
# it contains three repeats (rde_blobdataset, rde_measurements, rde_medadmin).
# Iterate over the deduplicated order so the manifest has one row per view.
rde_tables_unique = list(dict.fromkeys(rde_tables))

map_tables = ['map_address']

manifest_rows = []

print(f"target        : {target}")
print(f"staging       : {staging_schema}  (rebuild={REBUILD_STAGING})")
print(f"cohort        : {cohort_table}")
print(f"bcnb xwalk    : {bcnb_xwalk}")
print(f"DRY_RUN       : {DRY_RUN}")

In [0]:
# ------------------------------------------- staging: breast code lists ---
# Verbatim from the original notebook.

if REBUILD_STAGING:
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {staging_schema}")

    spark.sql(f"DROP TABLE IF EXISTS {staging_schema}.breast_patient_codes")

    spark.sql(f"""
    CREATE TABLE {staging_schema}.breast_patient_codes(
        code_type VARCHAR(100),
        code_value VARCHAR(400)
    )
    """)

    spark.sql(f"""
    INSERT INTO {staging_schema}.breast_patient_codes
    VALUES
    ('ICD-10_prefix', 'N60'),
    ('ICD-10_prefix', 'N61'),
    ('ICD-10_prefix', 'N62'),
    ('ICD-10_prefix', 'N63'),
    ('ICD-10_prefix', 'C50'),
    ('ICD-10_prefix', 'D24'),
    ('ICD-10_prefix', 'D05'),
    ('ICD-10_prefix', 'D486'),
    ('ICD-10_prefix', 'Z123'),
    ('ICD-10_prefix', 'Z901'),
    ('ICD-10_prefix', 'Z853'),
    ('OPCS-4_prefix', 'B27'),
    ('OPCS-4_prefix', 'B28'),
    ('OPCS-4_prefix', 'B29'),
    ('OPCS-4_prefix', 'B30'),
    ('OPCS-4_prefix', 'B31'),
    ('OPCS-4_prefix', 'B32'),
    ('OPCS-4_prefix', 'B33'),
    ('OPCS-4_prefix', 'B34'),
    ('OPCS-4_prefix', 'B35'),
    ('OPCS-4_prefix', 'B36'),
    ('OPCS-4_prefix', 'B37'),
    ('OPCS-4_prefix', 'B38'),
    ('OPCS-4_prefix', 'B39'),
    ('OPCS-4_prefix', 'B40'),
    ('OPCS-4_prefix', 'B41'),
    ('OPCS-4_prefix', 'U18'),
    ('OPCS-4_prefix', 'Z15')
    """)

    print("breast_patient_codes rebuilt")
else:
    print("staging rebuild skipped (DRY_RUN) - reading existing prod staging")

In [0]:
# ------------------------------------------------ staging: order signal ---
# Verbatim from the original notebook.

if REBUILD_STAGING:
    spark.sql(f"DROP TABLE IF EXISTS {staging_schema}.breast_patient_order_list_202406")

    spark.sql(f"""
    CREATE TABLE {staging_schema}.breast_patient_order_list_202406(
        PERSON_ID BIGINT,
        ORDER_ID BIGINT,
        CATALOG_CD BIGINT,
        ORDER_MNEMONIC VARCHAR(400),
        ORDER_DT_TM TIMESTAMP,
        SRC_TABLE VARCHAR(100)
    )
    """)

    spark.sql(f"""
    WITH cd AS (
        SELECT *
        FROM (
            VALUES
            (6180043), -- US Breast Rt
            (6181432), -- US Guided core biopsy breast Lt
            (6182217), -- US Breast Lt
            (6183363), -- US Guided core biopsy breast Rt
            (6183590) -- US Breast Both
        ) AS tmp(CATALOG_CD)
    )
    INSERT INTO {staging_schema}.breast_patient_order_list_202406
    SELECT
        PERSON_ID,
        ORDER_ID,
        o.CATALOG_CD,
        ORDER_MNEMONIC,
        o.ORIG_ORDER_DT_TM,
        'MILL_DIR_ORDERS'
    FROM 4_prod.raw.MILL_ORDERS AS o
    INNER JOIN cd
        ON o.CATALOG_CD = cd.CATALOG_CD
    """)

    print("breast_patient_order_list_202406 rebuilt")

In [0]:
# -------------------------------------------- staging: appointment signal ---
# Verbatim from the original notebook.

if REBUILD_STAGING:
    spark.sql(f"DROP TABLE IF EXISTS {staging_schema}.breast_patient_opa_list_202406")

    spark.sql(f"""
    CREATE TABLE {staging_schema}.breast_patient_opa_list_202406(
        person_id BIGINT,
        src_table VARCHAR(200),
        src_id VARCHAR(200),
        src_id_col VARCHAR(100),
        src_event_dt_tm TIMESTAMP,
        src_code_value VARCHAR(100)
    )
    """)

    spark.sql(f"""
    INSERT INTO {staging_schema}.breast_patient_opa_list_202406
      SELECT
        e.PERSON_ID,
        'MILL_SCH_APPT',
        CAST(sa.SCH_APPT_ID AS STRING),
        'SCH_APPT_ID',
        COALESCE(sa.BEG_DT_TM, sa.END_DT_TM),
        NULL
      FROM 4_prod.raw.mill_sch_appt AS sa
      INNER JOIN 4_prod.raw.mill_encounter AS e
        ON e.ENCNTR_ID = sa.ENCNTR_ID
      WHERE
        sa.DESCRIPTION ILIKE '%breast%'
        AND sa.ENCNTR_ID IS NOT NULL AND sa.ENCNTR_ID <> 0
        AND COALESCE(sa.BEG_DT_TM, sa.END_DT_TM) >= '2010-01-01'
        AND COALESCE(sa.BEG_DT_TM, sa.END_DT_TM) <= current_timestamp()
    """)

    print("breast_patient_opa_list_202406 rebuilt")

In [0]:
# ------------------------------------- staging: diagnosis/procedure signal ---
# Verbatim from the original notebook, including the unused `icd` CTE in the
# second INSERT. Left as-is deliberately: the WHERE clause already restricts to
# ICD-10 rows, so behaviour is identical, and preserving the text guarantees
# the cohort matches what Ashitha has already worked against.

if REBUILD_STAGING:
    spark.sql(f"DROP TABLE IF EXISTS {staging_schema}.breast_patient_list_202406")

    spark.sql(f"""
    CREATE TABLE {staging_schema}.breast_patient_list_202406(
        person_id BIGINT,
        ehr_code_type VARCHAR(100),
        ehr_code_value VARCHAR(100),
        src_table VARCHAR(200),
        src_id VARCHAR(200),
        src_id_col VARCHAR(100),
        src_event_dt_tm TIMESTAMP,
        src_code_value VARCHAR(100)
    )
    """)

    spark.sql(f"""
    WITH opcs AS (
        SELECT *
        FROM {staging_schema}.breast_patient_codes
        WHERE code_type ILIKE 'OPCS-4%'
    )
    INSERT INTO {staging_schema}.breast_patient_list_202406
    SELECT
        e.person_id,
        c.code_type AS ehr_code_type,
        c.code_value,
        'MILL_DIR_PROCEDURE' AS src_table,
        p.procedure_id,
        'PROCEDURE_ID' AS src_id_col,
        e.BEG_EFFECTIVE_DT_TM,
        COALESCE(n.CONCEPT_CKI, n.SOURCE_IDENTIFIER)
    FROM 4_prod.raw.MILL_PROCEDURE AS p
    INNER JOIN 3_lookup.mill.MILL_NOMENCLATURE AS n
        ON p.NOMENCLATURE_ID = n.NOMENCLATURE_ID
    INNER JOIN opcs AS c
        ON LTRIM(RTRIM(SOURCE_IDENTIFIER)) ILIKE CONCAT(LTRIM(RTRIM(c.code_value)),'%')
    INNER JOIN 4_prod.raw.MILL_ENCOUNTER AS e
        ON p.ENCNTR_ID = e.ENCNTR_ID
    WHERE
        SOURCE_VOCABULARY_CD = 685812 -- OPCS4
        AND e.BEG_EFFECTIVE_DT_TM IS NOT NULL
    """)

    spark.sql(f"""
    WITH icd AS (
        SELECT *
        FROM {staging_schema}.breast_patient_codes
        WHERE code_type ILIKE 'ICD-10%'
    )
    INSERT INTO {staging_schema}.breast_patient_list_202406
    SELECT
        e.person_id,
        c.code_type AS ehr_code_type,
        c.code_value,
        'MILL_DIR_DIAGNOSIS' AS src_table,
        d.diagnosis_id,
        'DIAGNOSIS_ID' AS src_id_col,
        e.BEG_EFFECTIVE_DT_TM,
        COALESCE(n.CONCEPT_CKI, n.SOURCE_IDENTIFIER)
    FROM 4_prod.raw.MILL_DIAGNOSIS AS d
    INNER JOIN 3_lookup.mill.MILL_NOMENCLATURE AS n
        ON d.NOMENCLATURE_ID = n.NOMENCLATURE_ID
    INNER JOIN {staging_schema}.breast_patient_codes AS c
        ON LTRIM(RTRIM(n.SOURCE_IDENTIFIER)) ILIKE CONCAT(LTRIM(RTRIM(c.code_value)),'%')
    INNER JOIN 4_prod.raw.MILL_ENCOUNTER AS e
        ON d.ENCNTR_ID = e.ENCNTR_ID
    WHERE
        c.code_type = 'ICD-10_prefix'
        AND CONCEPT_CKI ILIKE 'ICD10WHO%'
        AND e.BEG_EFFECTIVE_DT_TM IS NOT NULL
    """)

    print("breast_patient_list_202406 rebuilt")

In [0]:
# ------------------------------------------------------ BCNB crosswalk ---
# Breast Cancer Now Biobank id -> PERSON_ID, resolved through the current
# Millennium NHS-number alias. Materialised rather than inlined so that the
# ~90 downstream views do not each re-scan the 5.7M row alias table.
#
# Measured 2026-08-18: 3,838 of 3,867 BCNB rows resolve (99.25%), strictly 1:1
# in both directions. The assertion below exists because a silent fan-out here
# would multiply rows in every single downstream view.

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {bcnb_xwalk.rsplit('.', 1)[0]}")

spark.sql(f"""
CREATE OR REPLACE TABLE {bcnb_xwalk} AS
WITH b AS (
    SELECT DISTINCT
        BCNB_No,
        regexp_replace(CAST(NHS_No AS STRING), '[^0-9]', '') AS nhs
    FROM 6_mgmt.cohorts.bcnb
    WHERE NHS_No IS NOT NULL
),
m AS (
    SELECT DISTINCT
        regexp_replace(ALIAS_VALUE, '[^0-9]', '') AS nhs,
        PERSON_ID
    FROM 4_prod.bronze.map_patient_identifier
    WHERE ALIAS_TYPE = 'NHS'
      AND CURRENT_IND
)
SELECT DISTINCT m.PERSON_ID, b.BCNB_No
FROM b
INNER JOIN m ON b.nhs = m.nhs
WHERE length(b.nhs) = 10
""")

_x = spark.sql(f"""
    SELECT count(*) AS pairs,
           count(DISTINCT PERSON_ID) AS persons,
           count(DISTINCT BCNB_No)  AS ids
    FROM {bcnb_xwalk}
""").collect()[0]

assert _x['pairs'] == _x['persons'] == _x['ids'], (
    f"BCNB crosswalk is not 1:1 ({_x['pairs']} pairs, {_x['persons']} persons, "
    f"{_x['ids']} ids). Joining it to the cohort would fan out every downstream "
    f"view. Resolve the duplicates before continuing."
)

_src = spark.sql("SELECT count(*) AS n FROM 6_mgmt.cohorts.bcnb").collect()[0]['n']
print(f"BCNB crosswalk: {_x['pairs']} of {_src} resolved to PERSON_ID "
      f"({100.0 * _x['pairs'] / _src:.2f}%), 1:1 confirmed")

In [0]:
# ----------------------------------------------------------- the cohort ---
# Cohort membership logic is unchanged from the original. The only addition is
# BCNB_No, carried as a LEFT JOIN so cardinality is provably untouched: the
# crosswalk is 1:1 and the join is left, therefore rows in == rows out.
# Verified 2026-08-18: 128,742 persons, 3,364 carrying a BCNB id.

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {cohort_table.rsplit('.', 1)[0]}")

spark.sql(f"""
CREATE OR REPLACE VIEW {cohort_table} AS
WITH cte AS (
      -- Extracted from MILL_SCH_APPT (instead of PI_CDE_OP_ATTENDANCE)
      SELECT DISTINCT PERSON_ID
      FROM {staging_schema}.breast_patient_opa_list_202406
      WHERE src_event_dt_tm >= '2010-01-01'
      UNION
      -- Extracted from MILL_DIR_DIAGNOSIS, MILL_DIR_PROCEDURE
      SELECT DISTINCT PERSON_ID
      FROM {staging_schema}.breast_patient_list_202406
      WHERE src_event_dt_tm >= '2010-01-01'
      UNION
      -- Extracted from MILL_DIR_ORDERS
      SELECT DISTINCT PERSON_ID
      FROM {staging_schema}.breast_patient_order_list_202406
      WHERE order_dt_tm >= '2010-01-01'
  )
  SELECT DISTINCT c.PERSON_ID, x.BCNB_No
  FROM cte c
  LEFT JOIN {bcnb_xwalk} x ON x.PERSON_ID = c.PERSON_ID
""")

_c = spark.sql(f"""
    SELECT count(*) AS persons,
           count(BCNB_No) AS with_bcnb
    FROM {cohort_table}
""").collect()[0]
print(f"cohort: {_c['persons']} persons, {_c['with_bcnb']} carrying a BCNB id")

In [0]:
# --------------------------------------------------------------- helpers ---

import re

# Deny-list for sources that carry no ig_risk / ig_severity tags, calibrated
# against what the tags already remove from rde_patient_demographics: risk 4
# strips Date_of_Birth, Date_of_Death, MRN, NHS_Number and Postcode, while
# Year_of_Birth, City, Ethnicity and Gender are kept.
#
# Clinical dates (diagnosis, MDT, treatment, referral) are NOT matched here and
# come through intact. GP and consultant fields are also kept - verified to hold
# surrogate codes (C6079437, 573) rather than names.
#
# Measured effect on the 2026-08-18 run: 63 columns removed from
# scr_tbldemographics (names, carer/NOK blocks, addresses, NHS number) and
# Full_Name / Hospital_Number / NHS_Number removed from scr_bivwdiagnosis and
# scr_bivwinvestigations. Lookups lost nothing but ADC_UPDT.
UNTAGGED_DENY_PATTERNS = [
    # names
    r'SURNAME', r'FORENAME', r'^L_TITLE$', r'PAT_PREF_NAME', r'_NAME$',
    # address
    r'ADDRESS', r'POSTCODE', r'_TOWN$', r'_COUNTRY$',
    # contact details, next of kin, carers
    r'PHONE', r'^CONTACT_DETAILS$', r'^NOK_', r'^CARER',
    # direct identifiers
    r'NHS_NUMBER', r'^NHS_No$', r'HOSPITAL_NUMBER', r'^MRN$',
    # dates of birth and death
    r'DATE_BIRTH', r'_DOB$', r'DATE_DEATH',
]


def get_columns_with_high_tags(schema_name, table_name):
    """
    Get columns with high ig_risk or ig_severity tags for any schema/table.
    Unchanged from the original notebook.
    """
    high_risk_columns = spark.sql(f"""
        SELECT column_name
        FROM 4_prod.information_schema.column_tags
        WHERE schema_name = '{schema_name}'
        AND table_name = '{table_name}'
        AND tag_name = 'ig_risk'
        AND tag_value > {max_ig_risk}
    """).toPandas()['column_name'].tolist()

    high_severity_columns = spark.sql(f"""
        SELECT column_name
        FROM 4_prod.information_schema.column_tags
        WHERE schema_name = '{schema_name}'
        AND table_name = '{table_name}'
        AND tag_name = 'ig_severity'
        AND tag_value > {max_ig_severity}
    """).toPandas()['column_name'].tolist()

    return high_risk_columns + high_severity_columns


def denied_by_pattern(table_name, all_columns):
    """
    Deny-list for untagged sources. Reference lookups (scr_ltbl*) are exempt:
    they hold no patient data, and applying a '_NAME$' pattern to them would
    strip the very descriptions that make a lookup useful.
    """
    if table_name.lower().startswith('scr_ltbl'):
        return []
    return [c for c in all_columns
            if any(re.search(p, c, re.IGNORECASE) for p in UNTAGGED_DENY_PATTERNS)]


def resolve_columns(catalog, schema, table_name, use_alias=None, untagged=False):
    """
    Return (select_list, dropped_columns) for a source table.

    untagged=False -> tag-driven filter, identical to the original notebook.
    untagged=True  -> notebook-local deny-list, for sources with no IG tags.
    """
    all_columns = spark.table(f"{catalog}.{schema}.{table_name}").columns

    if untagged:
        sensitive = denied_by_pattern(table_name, all_columns)
    else:
        sensitive = get_columns_with_high_tags(schema, table_name)

    dropped = sorted(set(sensitive) | (set(all_columns) & set(columns_to_exclude)))
    kept = sorted(set(all_columns) - set(dropped))

    if use_alias:
        select_list = ", ".join([f"{use_alias}.`{c}`" for c in kept])
    else:
        select_list = ", ".join([f"`{c}`" for c in kept])

    return select_list, dropped


def find_person_id_column(full_table_path):
    """
    Finds the person ID column in a table given its full path.
    Unchanged from the original notebook.
    """
    columns = spark.table(full_table_path).columns
    potential_columns = [
        'PERSON_ID', 'person_id', 'Person_ID', 'personid',
        'PERSONID', 'PersonID', 'participant_id', 'PARENT_ENTITY_ID'
    ]

    for col in potential_columns:
        if col in columns:
            return col

    for col in columns:
        col_lower = col.lower()
        if 'person' in col_lower and 'id' in col_lower:
            return col

    return None


def record(view_name, source, person_method, dropped, cohort_filtered):
    manifest_rows.append({
        'view_name': view_name,
        'source_table': source,
        'person_id_method': person_method,
        'cohort_filtered': cohort_filtered,
        'columns_dropped': ", ".join(dropped) if dropped else "",
        'n_columns_dropped': len(dropped),
    })


spark.sql(f"USE CATALOG {target_catalog}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {target}")

# Drop all existing views in the schema so a removed source cannot linger.
#
# The cohort and BCNB crosswalk are protected. In a production run they live in
# 6_mgmt.cohorts and are not in scope for this loop anyway, but under DRY_RUN
# they are redirected into the target schema - and dropping the cohort view
# here would break every view built below it.
protected_objects = {name.split('.')[-1].lower()
                     for name in (cohort_table, bcnb_xwalk)
                     if name.rsplit('.', 1)[0].lower() == target.lower()}

existing_views_df = spark.sql(f"SHOW VIEWS IN {target}")
if existing_views_df.count() > 0:
    for row in existing_views_df.collect():
        if row.viewName.lower() in protected_objects:
            print(f"Kept protected view: {target}.{row.viewName}")
            continue
        spark.sql(f"DROP VIEW IF EXISTS {target}.`{row.viewName}`")
        print(f"Dropped view: {target}.{row.viewName}")

In [0]:
# --------------------------------------------------------- rde_* views ---
# Unchanged behaviour, plus BCNB_No picked up from the cohort join that was
# already being performed.

for table in rde_tables_unique:
    person_id_col = find_person_id_column(f"4_prod.rde.{table}")
    columns, dropped = resolve_columns('4_prod', 'rde', table, use_alias='s')

    if person_id_col:
        spark.sql(f"""
        CREATE OR REPLACE VIEW {target}.{table}
        AS
        SELECT {columns}, c.BCNB_No
        FROM 4_prod.rde.{table} s
        INNER JOIN {cohort_table} c
        ON s.{person_id_col} = c.PERSON_ID
        """)
        record(table, f"4_prod.rde.{table}", f"native {person_id_col}", dropped, True)
    else:
        spark.sql(f"""
        CREATE OR REPLACE VIEW {target}.{table}
        AS
        SELECT {columns}
        FROM 4_prod.rde.{table} s
        """)
        record(table, f"4_prod.rde.{table}", "none", dropped, False)
        print(f"Warning: No person ID column found in rde.{table}. Creating view without cohort filtering.")

    print(f"Created view: {target}.{table}")

In [0]:
# --------------------------------------------------------- map_* views ---
#
# PARENT_ENTITY_ID is only a person key when PARENT_ENTITY_NAME says so.
# map_address holds 42,490 ORGANIZATION rows alongside 17M PERSON rows, and
# without the guard below 9 of them joined cohort PERSON_IDs by numeric
# coincidence - an organisation's address attached to a patient. The condition
# keys off the resolved column rather than the table name so any future map_*
# table with the same shape is covered too.

def process_and_create_views(tables, source_catalog, source_schema):
    for table in tables:
        full_table_path = f"{source_catalog}.{source_schema}.{table}"
        person_id_col = find_person_id_column(full_table_path)
        columns, dropped = resolve_columns(source_catalog, source_schema, table, use_alias='m')

        if person_id_col:
            spark.sql(f"""
            CREATE OR REPLACE VIEW {target}.{table}
            AS
            SELECT {columns}, c.BCNB_No
            FROM {full_table_path} m
            INNER JOIN {cohort_table} c
            ON m.{person_id_col} = c.PERSON_ID
            {"WHERE m.PARENT_ENTITY_NAME = 'PERSON'" if person_id_col == 'PARENT_ENTITY_ID' else ""}
            """)
            record(table, full_table_path, f"native {person_id_col}", dropped, True)
        else:
            spark.sql(f"""
            CREATE OR REPLACE VIEW {target}.{table}
            AS
            SELECT {columns}
            FROM {full_table_path} m
            """)
            record(table, full_table_path, "none", dropped, False)
            print(f"Warning: No person ID column found in {full_table_path}. Creating view without cohort filtering.")

        print(f"Created view: {target}.{table}")


process_and_create_views(map_tables, '4_prod', 'bronze')

In [0]:
# ------------------------------------------------ genetics / pathology ---
# Anchored on map_pathology_accession.canonical_person_id. The anchor is
# materialised (16.78M rows for the cohort, measured 2026-08-18) so the
# downstream views are cheap joins rather than repeated scans of a 171M row
# source.
#
# These bronze tables ARE fully ig_risk / ig_severity tagged, so they use the
# tag-driven filter with no special handling.

spark.sql(f"""
CREATE OR REPLACE TABLE {target}.xwalk_pathology_accession AS
SELECT
    a.pathology_accession_id,
    a.canonical_person_id AS PERSON_ID,
    c.BCNB_No
FROM 4_prod.bronze.map_pathology_accession a
INNER JOIN {cohort_table} c
    ON c.PERSON_ID = a.canonical_person_id
""")

_g = spark.sql(f"""
    SELECT count(*) AS accessions, count(DISTINCT PERSON_ID) AS persons
    FROM {target}.xwalk_pathology_accession
""").collect()[0]
print(f"pathology anchor: {_g['accessions']} accessions for {_g['persons']} cohort persons")

# These all carry pathology_accession_id and join the anchor directly.
#
# map_pathology_indication is deliberately NOT in this list. The whole 3.21B row
# source is tagged ig_release_status='dev_only' / research_qi_only=true, it is
# diagnosis-context-window annotation rather than genetics, and it contributed
# 401M rows - roughly 40x the next largest object in the extract. Add it back
# here once it is released.
genetics_direct = [
    'map_pathology_accession',
    'map_pathology_requested_test',
    'map_pathology_report',
    'map_pathology_genetic_test',
]

for table in genetics_direct:
    columns, dropped = resolve_columns('4_prod', 'bronze', table, use_alias='s')
    spark.sql(f"""
    CREATE OR REPLACE VIEW {target}.{table} AS
    SELECT {columns}, x.PERSON_ID, x.BCNB_No
    FROM 4_prod.bronze.{table} s
    INNER JOIN {target}.xwalk_pathology_accession x
        ON s.pathology_accession_id = x.pathology_accession_id
    """)
    record(table, f"4_prod.bronze.{table}",
           "map_pathology_accession.canonical_person_id", dropped, True)
    print(f"Created view: {target}.{table}")

# gene_tested and genetic_result hang off genetic_test, not off the accession.
for table in ['map_pathology_gene_tested', 'map_pathology_genetic_result']:
    columns, dropped = resolve_columns('4_prod', 'bronze', table, use_alias='s')
    spark.sql(f"""
    CREATE OR REPLACE VIEW {target}.{table} AS
    SELECT {columns}, x.PERSON_ID, x.BCNB_No
    FROM 4_prod.bronze.{table} s
    INNER JOIN 4_prod.bronze.map_pathology_genetic_test g
        ON s.genetic_test_id = g.genetic_test_id
    INNER JOIN {target}.xwalk_pathology_accession x
        ON g.pathology_accession_id = x.pathology_accession_id
    """)
    record(table, f"4_prod.bronze.{table}",
           "map_pathology_genetic_test -> accession", dropped, True)
    print(f"Created view: {target}.{table}")

# Convenience: the structured test record next to its narrative report.
# map_pathology_genetic_result is empty today, so this is currently the only
# route to actual findings.
spark.sql(f"""
CREATE OR REPLACE VIEW {target}.genetics_report AS
SELECT
    x.PERSON_ID,
    x.BCNB_No,
    g.genetic_test_id,
    g.assay_code,
    g.assay_name,
    g.method,
    g.panel_code,
    g.panel_version,
    g.analysis_context,
    g.overall_result_status,
    r.report_version_id,
    r.report_role,
    r.discipline,
    r.report_section,
    r.report_text,
    r.issued_dt,
    r.is_current
FROM 4_prod.bronze.map_pathology_genetic_test g
INNER JOIN {target}.xwalk_pathology_accession x
    ON g.pathology_accession_id = x.pathology_accession_id
LEFT JOIN 4_prod.bronze.map_pathology_report r
    ON g.report_version_id = r.report_version_id
""")
record('genetics_report',
       '4_prod.bronze.map_pathology_genetic_test + map_pathology_report',
       'map_pathology_accession.canonical_person_id', [], True)
print(f"Created view: {target}.genetics_report")

In [0]:
# --------------------------------------------------- SCR person bridges ---
# No ancil_scr table carries PERSON_ID. These seven cohort-scoped crosswalks
# resolve each keying convention in the register. Coverage measured 2026-08-18.

# PATIENT_ID -> PERSON_ID via the NHS number alias. 325,590 / 326,487 = 99.7%.
spark.sql(f"""
CREATE OR REPLACE TABLE {target}.xwalk_scr_patient AS
WITH d AS (
    SELECT DISTINCT
        PATIENT_ID,
        regexp_replace(CAST(N1_1_NHS_NUMBER AS STRING), '[^0-9]', '') AS nhs
    FROM 4_prod.ancil_scr.scr_tbldemographics
),
m AS (
    SELECT DISTINCT
        regexp_replace(ALIAS_VALUE, '[^0-9]', '') AS nhs,
        PERSON_ID
    FROM 4_prod.bronze.map_patient_identifier
    WHERE ALIAS_TYPE = 'NHS' AND CURRENT_IND
)
SELECT DISTINCT d.PATIENT_ID, c.PERSON_ID, c.BCNB_No
FROM d
INNER JOIN m ON d.nhs = m.nhs AND length(d.nhs) = 10
INNER JOIN {cohort_table} c ON c.PERSON_ID = m.PERSON_ID
""")

# CARE_ID -> PATIENT_ID -> PERSON_ID
spark.sql(f"""
CREATE OR REPLACE TABLE {target}.xwalk_scr_care AS
SELECT DISTINCT r.CARE_ID, p.PERSON_ID, p.BCNB_No
FROM 4_prod.ancil_scr.scr_tblmain_referrals r
INNER JOIN {target}.xwalk_scr_patient p ON p.PATIENT_ID = r.PATIENT_ID
""")

# PATHOLOGY_ID -> CARE_ID. 811 / 811 = 100%.
spark.sql(f"""
CREATE OR REPLACE TABLE {target}.xwalk_scr_pathology AS
SELECT DISTINCT m.PATHOLOGY_ID, c.PERSON_ID, c.BCNB_No
FROM 4_prod.ancil_scr.scr_tblmain_pathology m
INNER JOIN {target}.xwalk_scr_care c ON c.CARE_ID = m.CARE_ID
""")

# IMAGE_ID -> CARE_ID. 35 / 35 = 100%.
spark.sql(f"""
CREATE OR REPLACE TABLE {target}.xwalk_scr_imaging AS
SELECT DISTINCT m.IMAGE_ID, c.PERSON_ID, c.BCNB_No
FROM 4_prod.ancil_scr.scr_tblmain_imaging m
INNER JOIN {target}.xwalk_scr_care c ON c.CARE_ID = m.CARE_ID
""")

# SURGERY_ID -> definitive_treatment.TREATMENT_ID -> CARE_ID. 5,138 / 5,491 = 93.6%.
# TREATMENT_ID is the correct key: TREAT_ID yields 9,803 matches against 5,491
# source rows, i.e. it fans out.
spark.sql(f"""
CREATE OR REPLACE TABLE {target}.xwalk_scr_surgery AS
SELECT DISTINCT t.TREATMENT_ID, c.PERSON_ID, c.BCNB_No
FROM 4_prod.ancil_scr.scr_tbldefinitive_treatment t
INNER JOIN {target}.xwalk_scr_care c ON c.CARE_ID = t.CARE_ID
""")

# MDT_ID -> CARE_ID, via the breast MDT table.
#
# scr_tblbreast_mdt.MDT_ID is typed STRING while scr_tblmdt_attendance.MDT_ID is
# INT, and 20 of the 69,658 source values are malformed (up to 21 digits). Left
# implicit, Spark casts the string side and the whole join fails with
# CAST_INVALID_INPUT. Cast explicitly to INT and drop only the malformed keys so
# the bridge type matches its consumer.
spark.sql(f"""
CREATE OR REPLACE TABLE {target}.xwalk_scr_mdt AS
SELECT DISTINCT try_cast(m.MDT_ID AS INT) AS MDT_ID, c.PERSON_ID, c.BCNB_No
FROM 4_prod.ancil_scr.scr_tblbreast_mdt m
INNER JOIN {target}.xwalk_scr_care c ON c.CARE_ID = m.CARE_ID
WHERE try_cast(m.MDT_ID AS INT) IS NOT NULL
""")

# SomaticTestSetID -> colorectal output -> CareID. The only landed route to the
# somatic test detail table - and currently a dead one: SomaticTestSetID is
# 100% NULL across all 348 source rows, so this crosswalk and the
# scr_mbsomatictests view it feeds are both empty by construction. Both are
# kept schema-stable so they populate if the column is ever landed.
spark.sql(f"""
CREATE OR REPLACE TABLE {target}.xwalk_scr_somatic AS
SELECT DISTINCT o.SomaticTestSetID, c.PERSON_ID, c.BCNB_No
FROM 4_prod.ancil_scr.scr_somatictestingcolorectaloutput o
INNER JOIN {target}.xwalk_scr_care c ON c.CARE_ID = o.CareID
WHERE o.SomaticTestSetID IS NOT NULL
""")

for x in ['patient', 'care', 'pathology', 'imaging', 'surgery', 'mdt', 'somatic']:
    n = spark.sql(f"SELECT count(*) AS n FROM {target}.xwalk_scr_{x}").collect()[0]['n']
    print(f"xwalk_scr_{x}: {n} rows")

In [0]:
# ----------------------------------------------------------- SCR views ---
# Breast tables plus the shared spine plus every ltbl* lookup.
# (table, key column on the source, bridge crosswalk, key column on the bridge)

scr_routes = [
    # --- shared spine, patient level
    ('scr_tbldemographics',                'PATIENT_ID',       'patient',   'PATIENT_ID'),
    ('scr_tblmain_referrals',              'PATIENT_ID',       'patient',   'PATIENT_ID'),
    # --- shared spine, care level
    ('scr_tblmain_care_plan',              'CARE_ID',          'care',      'CARE_ID'),
    ('scr_tblmain_pathology',              'CARE_ID',          'care',      'CARE_ID'),
    ('scr_tblmain_imaging',                'CARE_ID',          'care',      'CARE_ID'),
    ('scr_tbldefinitive_treatment',        'CARE_ID',          'care',      'CARE_ID'),
    ('scr_tblinitial_assessment',          'CARE_ID',          'care',      'CARE_ID'),
    ('scr_tbltracking_comments',           'CARE_ID',          'care',      'CARE_ID'),
    ('scr_bivwdiagnosis',                  'CARE_ID',          'care',      'CARE_ID'),
    ('scr_bivwinvestigations',             'CARE_ID',          'care',      'CARE_ID'),
    # --- breast
    ('scr_tblreferral_breast',             'CARE_ID',          'care',      'CARE_ID'),
    ('scr_tblbreast_mdt',                  'CARE_ID',          'care',      'CARE_ID'),
    ('scr_tblpathology_breast',            'PATHOLOGY_ID',     'pathology', 'PATHOLOGY_ID'),
    ('scr_tblsurgery_breast',              'SURGERY_ID',       'surgery',   'TREATMENT_ID'),
    ('scr_tblbreast_imaging',              'IMAGING_ID',       'imaging',   'IMAGE_ID'),
    # --- MDT
    ('scr_tblmdt_attendance',              'MDT_ID',           'mdt',       'MDT_ID'),
    # --- somatic / genomics
    ('scr_somatictestingcolorectaloutput', 'CareID',           'care',      'CARE_ID'),
    ('scr_mbsomatictests',                 'SomaticTestSetID', 'somatic',   'SomaticTestSetID'),
]

# No patient link in the landed data - exposed unfiltered. These hold meeting
# metadata and code decodes, not patient records.
scr_unfiltered = [
    'scr_tblmdt_meetings',
    'scr_tblmdt_attendees',
    'scr_vwmbsomaticteststatusorder',
]

scr_unfiltered += [r.tableName for r in spark.sql(
    "SHOW TABLES IN 4_prod.ancil_scr LIKE 'scr_ltbl*'").collect()]

for table, src_key, bridge, bridge_key in scr_routes:
    columns, dropped = resolve_columns('4_prod', 'ancil_scr', table,
                                       use_alias='s', untagged=True)
    spark.sql(f"""
    CREATE OR REPLACE VIEW {target}.{table} AS
    SELECT {columns}, b.PERSON_ID, b.BCNB_No
    FROM 4_prod.ancil_scr.{table} s
    INNER JOIN {target}.xwalk_scr_{bridge} b
        ON s.`{src_key}` = b.`{bridge_key}`
    """)
    record(table, f"4_prod.ancil_scr.{table}",
           f"{src_key} -> xwalk_scr_{bridge}", dropped, True)
    if dropped:
        print(f"Created view: {target}.{table}  [dropped: {', '.join(dropped)}]")
    else:
        print(f"Created view: {target}.{table}")

for table in scr_unfiltered:
    columns, dropped = resolve_columns('4_prod', 'ancil_scr', table,
                                       use_alias='s', untagged=True)
    spark.sql(f"""
    CREATE OR REPLACE VIEW {target}.{table} AS
    SELECT {columns}
    FROM 4_prod.ancil_scr.{table} s
    """)
    record(table, f"4_prod.ancil_scr.{table}", "none (reference data)", dropped, False)
    print(f"Created view: {target}.{table}  [reference, unfiltered]")

# Deliberately NOT exposed: scr_bivwhnaconcerns and scr_bivwhnadistressscores.
# Both key only on CNSContactID and no CNS contact table has landed, so they
# cannot be tied to a person. They are patient-level data, so exposing them
# unfiltered would be wrong. Revisit when the contact table lands.
print("\nOmitted (no patient route): scr_bivwhnaconcerns, scr_bivwhnadistressscores")

In [0]:
# ------------------------------------------------- manifest and schema ---
# Answers the standing request for methodology, mappings and processing steps.

import pandas as pd

spark.createDataFrame(pd.DataFrame(manifest_rows)) \
     .write.mode('overwrite').option('overwriteSchema', 'true') \
     .saveAsTable(f"{target}.extract_manifest")
print(f"Created table: {target}.extract_manifest  ({len(manifest_rows)} objects)")

if COMPUTE_MANIFEST_COUNTS:
    counts = []
    for r in manifest_rows:
        try:
            n = spark.sql(f"SELECT count(*) AS n FROM {target}.`{r['view_name']}`") \
                     .collect()[0]['n']
        except Exception as e:
            n = -1
            print(f"count failed for {r['view_name']}: {e}")
        counts.append({'view_name': r['view_name'], 'row_count': n})
    spark.createDataFrame(pd.DataFrame(counts)) \
         .write.mode('overwrite').option('overwriteSchema', 'true') \
         .saveAsTable(f"{target}.extract_row_counts")
    print(f"Created table: {target}.extract_row_counts")

spark.sql(f"""
CREATE OR REPLACE VIEW {target}.schema AS
SELECT
    table_name,
    column_name,
    COALESCE(comment, '') as column_comment
FROM {target_catalog}.information_schema.columns
WHERE table_catalog = '{target_catalog}'
AND table_schema = '{target_schema}'
AND table_name != 'schema'
ORDER BY table_name, column_name
""")
print(f"Created schema view: {target}.schema")

# Drop the previous run's map_pathology_indication view if it is still present.
# It was exposed by the first production run and is no longer built.
spark.sql(f"DROP VIEW IF EXISTS {target}.map_pathology_indication")

In [0]:
# ------------------------------------------------------------- summary ---

print(f"DAC003 v2 complete -> {target}\n")
print(f"  cohort            : {_c['persons']} persons, {_c['with_bcnb']} with a BCNB id")
print(f"  rde_* views       : {len(rde_tables_unique)}")
print(f"  map_* views       : {len(map_tables)}")
print(f"  genetics views    : {len(genetics_direct) + 3}")
print(f"  SCR views         : {len(scr_routes)} cohort-filtered + {len(scr_unfiltered)} reference")
print(f"  crosswalk tables  : 8")
print(f"  manifest rows     : {len(manifest_rows)}")
print("\nKnown limitations (see ~/docs/2026-08-18-dac003-v2-design.md):")
print("  - map_pathology_indication is NOT exposed: dev_only release status,")
print("    401M rows, and diagnosis-context annotation rather than genetics")
print("  - map_pathology_genetic_result is currently EMPTY (parser not yet run);")
print("    report text via genetics_report is the only route to findings today")
print("  - scr_mbsomatictests and xwalk_scr_somatic are EMPTY by construction:")
print("    SomaticTestSetID is 100% NULL across all 348 source rows")
print("  - 474 resolvable BCNB patients fall outside the cohort and carry no row")
print("  - scr_tblsurgery_breast loses ~6.4% of rows with no TREATMENT_ID match")
print("  - 20 of 69,658 scr_tblbreast_mdt rows have a malformed MDT_ID and are")
print("    absent from xwalk_scr_mdt (they still appear in scr_tblbreast_mdt)")